# 03 - DML, History e Time Travel em Delta Lake

Este notebook demonstra operacoes transacionais em tabelas Delta Lake armazenadas no bucket **bronze** do MinIO.

A demonstracao principal usa a tabela `produtos` e o produto de teste `id = 999` para mostrar:

- `INSERT`
- `UPDATE`
- `DELETE`
- `DESCRIBE HISTORY`
- `TIME TRAVEL` com `versionAsOf`

## 1. Configuracao e SparkSession

In [ ]:
import os

from delta.tables import DeltaTable
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv(override=True)

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BRONZE_BUCKET = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

tables = ['clientes', 'produtos', 'pedidos', 'itens_pedido']
produto_teste_id = 999

spark = (
    SparkSession.builder
    .appName('DML Delta Lake Bronze')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession criada com suporte a Delta Lake e MinIO.')
print(f'Bucket bronze: {BRONZE_BUCKET}')

## 2. Registrar tabelas Delta do bronze no Spark SQL

In [ ]:
for table in tables:
    delta_path = f's3a://{BRONZE_BUCKET}/{table}'
    spark.sql(f'DROP TABLE IF EXISTS {table}')
    spark.sql(f"""
        CREATE TABLE {table}
        USING delta
        LOCATION '{delta_path}'
    """)

print('Tabelas Delta registradas no Spark SQL:')
spark.sql('SHOW TABLES').show(truncate=False)

## 3. Leitura inicial das tabelas Delta

In [ ]:
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)

for table in tables:
    count = spark.sql(f'SELECT COUNT(*) AS total FROM {table}').collect()[0]['total']
    print(f'{table:<15} {count:>10}')

print('\nAmostra da tabela produtos:')
spark.sql('SELECT * FROM produtos ORDER BY id LIMIT 10').show(truncate=False)

## 4. Preparar produto de teste

In [ ]:
spark.sql(f'DELETE FROM produtos WHERE id = {produto_teste_id}')

produto_path = f's3a://{BRONZE_BUCKET}/produtos'
dt_produtos = DeltaTable.forPath(spark, produto_path)
history_before = dt_produtos.history().select('version').orderBy('version').collect()
versao_base = history_before[-1]['version']

print(f'Produto {produto_teste_id} removido caso existisse de execucoes anteriores.')
print(f'Versao base antes do INSERT: {versao_base}')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

## 5. INSERT - inserir produto id 999

In [ ]:
spark.sql(f"""
    INSERT INTO produtos VALUES
    ({produto_teste_id}, 'Produto Delta Teste', 'Demo Delta Lake', 99.90, 10, true)
""")

print('Produto id 999 inserido:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_insert = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos INSERT: {versao_insert}')

## 6. UPDATE - atualizar produto id 999

In [ ]:
spark.sql(f"""
    UPDATE produtos
    SET nome_produto = 'Produto Delta Teste Atualizado',
        preco = 149.90,
        estoque = 25,
        ativo = false
    WHERE id = {produto_teste_id}
""")

print('Produto id 999 atualizado:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_update = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos UPDATE: {versao_update}')

## 7. DELETE - deletar produto id 999

In [ ]:
spark.sql(f'DELETE FROM produtos WHERE id = {produto_teste_id}')

print('Produto id 999 apos DELETE:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_delete = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos DELETE: {versao_delete}')

## 8. HISTORY - historico transacional da tabela produtos

In [ ]:
print('Historico Delta da tabela produtos:')
spark.sql(f'DESCRIBE HISTORY delta.`{produto_path}`') \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

## 9. TIME TRAVEL - consultar versoes anteriores

In [ ]:
print(f'Versao base ({versao_base}) - antes do produto 999:')
spark.read.format('delta') \
    .option('versionAsOf', versao_base) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao INSERT ({versao_insert}) - produto 999 inserido:')
spark.read.format('delta') \
    .option('versionAsOf', versao_insert) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao UPDATE ({versao_update}) - produto 999 atualizado:')
spark.read.format('delta') \
    .option('versionAsOf', versao_update) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao DELETE ({versao_delete}) - produto 999 deletado:')
spark.read.format('delta') \
    .option('versionAsOf', versao_delete) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

## 10. Resumo final

In [ ]:
print('=' * 70)
print('RESUMO DAS OPERACOES DELTA LAKE')
print('=' * 70)
print(f'Produto de teste: id {produto_teste_id}')
print(f'1. INSERT executado na versao {versao_insert}')
print(f'2. UPDATE executado na versao {versao_update}')
print(f'3. DELETE executado na versao {versao_delete}')
print('4. HISTORY exibiu as operacoes WRITE, UPDATE e DELETE')
print('5. TIME TRAVEL mostrou o estado da tabela antes e depois de cada operacao')
print('=' * 70)

## 11. Encerrar Spark

In [ ]:
spark.stop()
print('SparkSession finalizada.')